# Performance Benchmarks of `rhs_hyperbolic!` Functions on CPU and GPU

Start this notebook with one Julia thread and a functional CUDA GPU. The CPU/GPU plots use the shared Float64 case configurations in `cases/` and seeded random inputs.


In [ ]:
# Reuse the case definitions, random inputs, and GPU measurement harness.
import Pkg
Pkg.activate(@__DIR__)
Pkg.develop(path = dirname(@__DIR__))
Pkg.instantiate()
include("runbenchmarks.jl")
using Statistics: mean, median

# Plotting is optional and is not needed by the automated benchmarks.
Pkg.add("Plots")
using Plots

function benchmark_cpu_gpu(name; seconds = 2.0, samples = 100, seed = 123)
    case = only(filter(c -> c.name == name, rhs_cases()))
    config = case.build()
    semi_gpu = make_problem(case)
    solver = DGSEM(; polydeg = config.polydeg, surface_flux = config.surface_flux,
                   volume_integral = config.volume_integral, RealT = Float64)
    semi = SemidiscretizationHyperbolic(semi_gpu.mesh, semi_gpu.equations, nothing, solver;
                                        boundary_conditions = config.boundary_conditions,
                                        source_terms = config.source_terms)
    (; mesh, equations, boundary_conditions, source_terms, solver, cache) = semi
    u = random_input(equations, solver, cache, seed)
    du = zeros(size(u))
    t = 0.0
    Trixi.rhs_hyperbolic!(nothing, du, u, t, mesh, equations, boundary_conditions,
                          source_terms, solver, cache)
    all(isfinite, du) || error("Non-finite CPU RHS in $name")
    cpu_trial = @benchmark Trixi.rhs_hyperbolic!(nothing, $du, $u, $t, $mesh, $equations,
                                                 $boundary_conditions, $source_terms, $solver, $cache) setup=(fill!($du, 0.0)) evals=1 seconds=seconds samples=samples
    gpu_result = benchmark_rhs(case; seconds, samples, seed, semi = semi_gpu)
    return cpu_trial, (; times = gpu_result["samples_ns"]), gpu_result["dofs_per_field"]
end


We also need to create a new subdirectory under benchmark to store the plots (optional).

In [ ]:
mkpath("plots")

## Way 1: Run and plot each benchmark case individually

The helper builds both solvers from the same case configuration and reuses the GPU warm-up and measurement function from `runbenchmarks.jl`. Times are plotted as mean and median, alongside DOFs per field.


In [ ]:
# Run benchmarks
cpu_trial, gpu_trial, dofs = benchmark_cpu_gpu("advection_basic_1d")

advec_1d_cpu = cpu_trial
advec_1d_gpu = gpu_trial
advec_1d_dofs = dofs

cpu_trial, gpu_trial, dofs = benchmark_cpu_gpu("advection_basic_2d")

advec_2d_cpu = cpu_trial
advec_2d_gpu = gpu_trial
advec_2d_dofs = dofs

cpu_trial, gpu_trial, dofs = benchmark_cpu_gpu("advection_basic_3d") 

advec_3d_cpu = cpu_trial
advec_3d_gpu = gpu_trial
advec_3d_dofs = dofs

# Medians
cpu_median_vals = [
    median(advec_1d_cpu.times) * 10^-3,
    median(advec_2d_cpu.times) * 10^-3,
    median(advec_3d_cpu.times) * 10^-3
]
gpu_median_vals = [
    median(advec_1d_gpu.times) * 10^-3,
    median(advec_2d_gpu.times) * 10^-3,
    median(advec_3d_gpu.times) * 10^-3
]
# Means
cpu_mean_vals = [
    mean(advec_1d_cpu.times) * 10^-3,
    mean(advec_2d_cpu.times) * 10^-3,
    mean(advec_3d_cpu.times) * 10^-3
]
gpu_mean_vals = [
    mean(advec_1d_gpu.times) * 10^-3,
    mean(advec_2d_gpu.times) * 10^-3,
    mean(advec_3d_gpu.times) * 10^-3
]
# Standard deviations
# cpu_stdev_vals = [
#     sqrt(var(advec_1d_cpu.times) * 10^-6),
#     sqrt(var(advec_2d_cpu.times) * 10^-6),
#     sqrt(var(advec_3d_cpu.times) * 10^-6)
# ]
# gpu_stdev_vals = [
#     sqrt(var(advec_1d_gpu.times) * 10^-6),
#     sqrt(var(advec_2d_gpu.times) * 10^-6),
#     sqrt(var(advec_3d_gpu.times) * 10^-6)
# ]

# DOFs (with base log10)
log_dofs_vals = [
    log10(advec_1d_dofs),
    log10(advec_2d_dofs),
    log10(advec_3d_dofs)
]

dims = ["1D", "2D", "3D"]

# Plot DOFs
plt = plot(dims, log_dofs_vals;
    seriestype=:line,
    fillrange=0,
    fillalpha=0.2,
    linewidth=0,
    label="log₁₀(DOFs)",
    ylabel="log₁₀(DOFs)",
    xlabel="Dimension",
    legend=:bottomright,
    title="Linear Advection Equation (Basic)",
    size=(500, 500)
)

plt2 = twinx(plt)
# Mean with deviation - CPU
plot!(plt2, dims, cpu_mean_vals;
    # yerror=cpu_stdev_vals,
    seriestype=:line,
    marker=:circle,
    label="Mean time CPU",
    ylabel="Time (μs)",
    legend=:topleft,
    linewidth=1.5,
    linestyle=:solid,
    color=:darkorange2
)
# Mean mith deviation - GPU
plot!(plt2, dims, gpu_mean_vals;
    # yerror=gpu_stdev_vals,
    seriestype=:line,
    marker=:circle,
    label="Mean time GPU",
    linewidth=1.5,
    linestyle=:solid,
    color=:green
)
# Median - CPU
plot!(plt2, dims, cpu_median_vals;
    marker=:diamond,
    seriestype=:line,
    label="Median time CPU",
    linewidth=1.5,
    linestyle=:dash,
    color=:orange
)
# Median - GPU
plot!(plt2, dims, gpu_median_vals;
    marker=:diamond,
    seriestype=:line,
    label="Median time GPU",
    linewidth=1.5,
    linestyle=:dash,
    color=:yellowgreen
)

# Save figure
savefig(plt, "./plots/plot_advec.png")

In [ ]:
# Run benchmarks
cpu_trial, gpu_trial, dofs = benchmark_cpu_gpu("advection_mortar_2d")   

advecmtr_2d_cpu = cpu_trial
advecmtr_2d_gpu = gpu_trial
advecmtr_2d_dofs = dofs

cpu_trial, gpu_trial, dofs = benchmark_cpu_gpu("advection_mortar_3d")

advecmtr_3d_cpu = cpu_trial
advecmtr_3d_gpu = gpu_trial
advecmtr_3d_dofs = dofs

# Medians
cpu_median_vals = [
    median(advecmtr_2d_cpu.times) * 10^-3,
    median(advecmtr_3d_cpu.times) * 10^-3
]
gpu_median_vals = [
    median(advecmtr_2d_gpu.times) * 10^-3,
    median(advecmtr_3d_gpu.times) * 10^-3
]
# Means
cpu_mean_vals = [
    mean(advecmtr_2d_cpu.times) * 10^-3,
    mean(advecmtr_3d_cpu.times) * 10^-3
]
gpu_mean_vals = [
    mean(advecmtr_2d_gpu.times) * 10^-3,
    mean(advecmtr_3d_gpu.times) * 10^-3
]
# Standard deviations
# cpu_stdev_vals = [
#     sqrt(var(advecmtr_2d_cpu.times) * 10^-6),
#     sqrt(var(advecmtr_3d_cpu.times) * 10^-6)
# ]
# gpu_stdev_vals = [
#     sqrt(var(advecmtr_2d_gpu.times) * 10^-6),
#     sqrt(var(advecmtr_3d_gpu.times) * 10^-6)
# ]

# DOFs (with base log10)
log_dofs_vals = [
    log10(advecmtr_2d_dofs),
    log10(advecmtr_3d_dofs)
]

dims = ["2D", "3D"]

# Plot DOFs
plt = plot(dims, log_dofs_vals;
    seriestype=:line,
    fillrange=0,
    fillalpha=0.2,
    linewidth=0,
    label="log₁₀(DOFs)",
    ylabel="log₁₀(DOFs)",
    xlabel="Dimension",
    legend=:bottomright,
    title="Linear Advection Equation (Mortar)",
    size=(500, 500),
    xlims=(-0.1, 2.1)
)

plt2 = twinx(plt)
# Mean with deviation - CPU
plot!(plt2, dims, cpu_mean_vals;
    # yerror=cpu_stdev_vals,
    seriestype=:line,
    marker=:circle,
    label="Mean time CPU",
    ylabel="Time (μs)",
    legend=:topleft,
    linewidth=1.5,
    linestyle=:solid,
    color=:darkorange2,
    xlims=(-0.1, 2.1)
)
# Mean mith deviation - GPU
plot!(plt2, dims, gpu_mean_vals;
    # yerror=gpu_stdev_vals,
    seriestype=:line,
    marker=:circle,
    label="Mean time GPU",
    linewidth=1.5,
    linestyle=:solid,
    color=:green
)
# Median - CPU
plot!(plt2, dims, cpu_median_vals;
    marker=:diamond,
    seriestype=:line,
    label="Median time CPU",
    linewidth=1.5,
    linestyle=:dash,
    color=:orange
)
# Median - GPU
plot!(plt2, dims, gpu_median_vals;
    marker=:diamond,
    seriestype=:line,
    label="Median time GPU",
    linewidth=1.5,
    linestyle=:dash,
    color=:yellowgreen
)

# Save figure
savefig(plt, "./plots/plot_advecmtr.png")

In [ ]:
# Run benchmarks
cpu_trial, gpu_trial, dofs = benchmark_cpu_gpu("euler_shock_1d")

eulersk_1d_cpu = cpu_trial
eulersk_1d_gpu = gpu_trial
eulersk_1d_dofs = dofs

cpu_trial, gpu_trial, dofs = benchmark_cpu_gpu("euler_shock_2d") 

eulersk_2d_cpu = cpu_trial
eulersk_2d_gpu = gpu_trial
eulersk_2d_dofs = dofs

cpu_trial, gpu_trial, dofs = benchmark_cpu_gpu("euler_shock_3d")

eulersk_3d_cpu = cpu_trial
eulersk_3d_gpu = gpu_trial
eulersk_3d_dofs = dofs

# Medians
cpu_median_vals = [
    median(eulersk_1d_cpu.times) * 10^-3,
    median(eulersk_2d_cpu.times) * 10^-3,
    median(eulersk_3d_cpu.times) * 10^-3
]
gpu_median_vals = [
    median(eulersk_1d_gpu.times) * 10^-3,
    median(eulersk_2d_gpu.times) * 10^-3,
    median(eulersk_3d_gpu.times) * 10^-3
]
# Means
cpu_mean_vals = [
    mean(eulersk_1d_cpu.times) * 10^-3,
    mean(eulersk_2d_cpu.times) * 10^-3,
    mean(eulersk_3d_cpu.times) * 10^-3
]
gpu_mean_vals = [
    mean(eulersk_1d_gpu.times) * 10^-3,
    mean(eulersk_2d_gpu.times) * 10^-3,
    mean(eulersk_3d_gpu.times) * 10^-3
]
# Standard deviations
# cpu_stdev_vals = [
#     sqrt(var(eulersk_1d_cpu.times) * 10^-6),
#     sqrt(var(eulersk_2d_cpu.times) * 10^-6),
#     sqrt(var(eulersk_3d_cpu.times) * 10^-6)
# ]
# gpu_stdev_vals = [
#     sqrt(var(eulersk_1d_gpu.times) * 10^-6),
#     sqrt(var(eulersk_2d_gpu.times) * 10^-6),
#     sqrt(var(eulersk_3d_gpu.times) * 10^-6)
# ]

# DOFs (with base log10)
log_dofs_vals = [
    log10(eulersk_1d_dofs),
    log10(eulersk_2d_dofs),
    log10(eulersk_3d_dofs)
]

dims = ["1D", "2D", "3D"]

# Plot DOFs
plt = plot(dims, log_dofs_vals;
    seriestype=:line,
    fillrange=0,
    fillalpha=0.2,
    linewidth=0,
    label="log₁₀(DOFs)",
    ylabel="log₁₀(DOFs)",
    xlabel="Dimension",
    legend=:bottomright,
    title="Euler Equations (Shock Capturing)",
    size=(500, 500)
)

plt2 = twinx(plt)
# Mean with deviation - CPU
plot!(plt2, dims, cpu_mean_vals;
    # yerror=cpu_stdev_vals,
    seriestype=:line,
    marker=:circle,
    label="Mean time CPU",
    ylabel="Time (μs)",
    legend=:topleft,
    linewidth=1.5,
    linestyle=:solid,
    color=:darkorange2
)
# Mean mith deviation - GPU
plot!(plt2, dims, gpu_mean_vals;
    # yerror=gpu_stdev_vals,
    seriestype=:line,
    marker=:circle,
    label="Mean time GPU",
    linewidth=1.5,
    linestyle=:solid,
    color=:green
)
# Median - CPU
plot!(plt2, dims, cpu_median_vals;
    marker=:diamond,
    seriestype=:line,
    label="Median time CPU",
    linewidth=1.5,
    linestyle=:dash,
    color=:orange
)
# Median - GPU
plot!(plt2, dims, gpu_median_vals;
    marker=:diamond,
    seriestype=:line,
    label="Median time GPU",
    linewidth=1.5,
    linestyle=:dash,
    color=:yellowgreen
)

savefig(plt, "./plots/plot_eulersk.png")

Only three groups are shown above; additional case configurations are in `benchmark/cases/`.


## Way 2: Benchmark and plot all examples in batch, in a single run
In this approach, all benchmarks are run in a single session, making the process faster and more convenient than running each benchmark individually.

Again, we plot the mean and median times for both CPU and GPU, and include the degrees of freedom (DOFs) in the benchmark plots.

In [ ]:
# Define examples
examples = [
    # ("Linear Advection Equation (Basic)", [
    #     "advection_basic_1d.jl",
    #     "advection_basic_2d.jl",
    #     "advection_basic_3d.jl"
    # ]),
    # ("Linear Advection Equation (Mortar)", [
    #     "advection_mortar_2d.jl",
    #     "advection_mortar_3d.jl"
    # ]),
    # ("Euler Equations (Entropy Conservative)", [
    #     "euler_ec_1d.jl",
    #     "euler_ec_2d.jl",
    #     "euler_ec_3d.jl"
    # ]),
    # ("Euler Equations (Shock Capturing)", [
    #     "euler_shock_1d.jl",
    #     "euler_shock_2d.jl",
    #     "euler_shock_3d.jl"
    # ]),
    ("MHD Equations (Entropy Conservative)", [
        "mhd_ec_1d.jl",
        "mhd_ec_2d.jl",
        "mhd_ec_3d.jl"
    ]),
    ("MHD Equations (Alfven Wave)", [
        "mhd_alfven_wave_1d.jl",
        "mhd_alfven_wave_2d.jl",
        "mhd_alfven_wave_3d.jl"
    ]),
    # ("MHD Equations (Alfven Wave Mortar)", [
    #     "mhd_alfven_wave_mortar_2d.jl",
    #     "mhd_alfven_wave_mortar_3d.jl"
    # ]),
    # ("Hyperbolic Diffusion (Non-periodic)", [
    #     "hypdiff_nonperiodic_1d.jl",
    #     "hypdiff_nonperiodic_2d.jl",
    #     "hypdiff_nonperiodic_3d.jl"
    # ]),
    # ("Shallow Water (Entropy Conservative)", [
    #     "shallowwater_ec_1d.jl",
    #     "shallowwater_ec_2d.jl",
    # ]),
    # ("Shallow water (Source Terms)", [
    #     "shallowwater_source_terms_1d.jl",
    #     "shallowwater_source_terms_2d.jl",
    # ])
]

# Define dimensions
dims = ["1D", "2D", "3D"]
# dims = ["1D", "2D"]
# dims = ["2D", "3D"]

# Initialize arrays
cpu_medians = zeros(length(examples), length(dims))
gpu_medians = zeros(length(examples), length(dims))
cpu_means = zeros(length(examples), length(dims))
gpu_means = zeros(length(examples), length(dims))
# cpu_stdevs = zeros(length(examples), length(dims))
# gpu_stdevs = zeros(length(examples), length(dims))

# Add DOFs
dofs_array = zeros(length(examples), length(dims))

# Loop over examples
for (i, (name, scripts)) in enumerate(examples)
    for (j, script) in enumerate(scripts)
        cpu_trial, gpu_trial, dofs = benchmark_cpu_gpu(splitext(script)[1])

        # Convert to μs
        cpu_medians[i, j] = median(cpu_trial.times) * 10^-3
        gpu_medians[i, j] = median(gpu_trial.times) * 10^-3
        cpu_means[i, j] = mean(cpu_trial.times) * 10^-3
        gpu_means[i, j] = mean(gpu_trial.times) * 10^-3
        # cpu_stdevs[i, j] = sqrt(var(cpu_trial.times) * 10^-6)
        # gpu_stdevs[i, j] = sqrt(var(gpu_trial.times) * 10^-6)

        # DOFs
        dofs_array[i, j] = log10(dofs)
    end

    # Extract values
    cpu_mean_vals = cpu_means[i, :]
    gpu_mean_vals = gpu_means[i, :]
    cpu_median_vals = cpu_medians[i, :]
    gpu_median_vals = gpu_medians[i, :]
    # cpu_stdev_vals = cpu_stdevs[i, :]
    # gpu_stdev_vals = gpu_stdevs[i, :]

    # Extract DOFs
    log_dofs_vals = dofs_array[i, :]

    # Plot DOFs
    plt = plot(dims, log_dofs_vals;
        seriestype=:line,
        fillrange=0,
        fillalpha=0.2,
        linewidth=0,
        label="log₁₀(DOFs)",
        ylabel="log₁₀(DOFs)",
        xlabel="Dimension",
        legend=:bottomright,
        title=name,
        size=(500, 500),
        # xlims=(-0.1, 2.1) # for 1D and 2D, or 2D and 3D only
    )

    plt2 = twinx(plt)
    # Plot mean with deviation - CPU
    plot!(plt2, dims, cpu_mean_vals;
        # yerror=cpu_stdev_vals,
        seriestype=:line,
        marker=:circle,
        label="Mean time CPU",
        ylabel="Time (μs)",
        legend=:topleft,
        linewidth=1.5,
        linestyle=:solid,
        color=:darkorange2,
        # xlims=(-0.1, 2.1) # for 1D and 2D, or 2D and 3D only
    )
    # Plot mean mith deviation - GPU
    plot!(plt2, dims, gpu_mean_vals;
        # yerror=gpu_stdev_vals,
        seriestype=:line,
        marker=:circle,
        label="Mean time GPU",
        linewidth=1.5,
        linestyle=:solid,
        color=:green
    )
    # Plot median - CPU
    plot!(plt2, dims, cpu_median_vals;
        marker=:diamond,
        seriestype=:line,
        label="Median time CPU",
        linewidth=1.5,
        linestyle=:dash,
        color=:orange
    )
    # Plot median - GPU
    plot!(plt2, dims, gpu_median_vals;
        marker=:diamond,
        seriestype=:line,
        label="Median time GPU",
        linewidth=1.5,
        linestyle=:dash,
        color=:yellowgreen
    )

    # Save figures
    fname = "plots/" * replace(lowercase(name), " " => "_", 
                                                "(" => "", 
                                                ")" => "", 
                                                "." => "", 
                                                "," => "") * ".png"
    savefig(plt, fname)
    println("Saved: ", fname)
end


Add a configuration file under `cases/` to make a new case available to both the automated benchmarks and this notebook.
